# Retrain on Colab

Rebuilds the model with the leak-free pipeline:
- imputation and frequency maps are fitted on train only
- the latest 20% of train is held out **before** SMOTE as validation
- Optuna tunes on validation, and the threshold is picked on validation
- the test set is scored once, at the end

Before running, add a Colab secret named `KAGGLE_API_TOKEN` (key icon in the left bar) and accept the competition rules at kaggle.com/c/ieee-fraud-detection/rules. A standard CPU runtime is enough; High-RAM helps.

In [1]:
!git clone -q https://github.com/Shashank17singh/Agentic-Fraud-Sentinel.git
%cd Agentic-Fraud-Sentinel
!git log --oneline -1
!pip install -q -r requirements.txt kaggle

/content/Agentic-Fraud-Sentinel
0bb50fe (HEAD -> main, origin/main, origin/HEAD) Train XGBoost on the GPU when one is available


In [2]:
import os

from google.colab import userdata

os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
!kaggle competitions download -c ieee-fraud-detection -p data/raw -q
!unzip -oq data/raw/ieee-fraud-detection.zip train_transaction.csv train_identity.csv -d data/raw
!ls -la data/raw

total 814216
drwxr-xr-x 2 root root      4096 Nov 22 21:55 .
drwxr-xr-x 3 root root      4096 Nov 22 21:55 ..
-rw-r--r-- 1 root root 123856947 Dec 11  2019 ieee-fraud-detection.zip
-rw-r--r-- 1 root root  26529680 Dec 11  2019 train_identity.csv
-rw-r--r-- 1 root root 683351067 Dec 11  2019 train_transaction.csv


In [3]:
import sys

# Same layout as the local notebooks: outputs land in notebooks/data/, where the API looks.
%cd notebooks
sys.path.insert(0, "..")

from src.data.preprocessing import run_preprocessing_pipeline

X_train, X_val, X_test, Y_train, Y_val, Y_test = run_preprocessing_pipeline(
    raw_transaction_path="../data/raw/train_transaction.csv",
    raw_identity_path="../data/raw/train_identity.csv",
)

/content/Agentic-Fraud-Sentinel/notebooks
Loading data...
Dropping 6 high-missing transaction cols
Dropping 9 high-missing identity cols
Loading transaction file...
Transaction: 1257.5 MB
Loading identity file...
Identity: 123.3 MB
Joined shape: (590540, 419)

Engineering features...

Splitting...
Split point: 12,192,854 seconds
Train: 472,432 rows | Fraud rate: 3.514%
Test:  118,108 rows  | Fraud rate: 3.441%

Frequency encoding...

Imputing (medians from train only)...

Encoding categoricals...
Encoded 29 categorical columns

Separating features and target...
X_train: 472,432 rows | Y_train: 472,432 rows
X_test:  118,108 rows  | Y_test:  118,108 rows
X_val:   94,487 rows (held out before SMOTE)

Applying SMOTE...
X_train rows: 377,945 | Y_train rows: 377,945
Before SMOTE � fraud: 12,895 / 377,945 (3.412%)
After SMOTE  � fraud: 36,505 / 401,555 (9.091%)

Saving...
Saved to data/processed/


In [4]:
from src.models.evaluate import evaluate_model, log_results, select_threshold
from src.models.train import train_xgboost_baseline

baseline = train_xgboost_baseline(X_train, Y_train)
baseline_results = evaluate_model(baseline, X_test, Y_test, model_name="xgb_baseline")
log_results(baseline_results)

/usr/local/lib/python3.13/dist-packages/xgboost/core.py:158: UserWarning: [21:57:50] WARNING: /workspace/src/common/error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  warnings.warn(smsg, UserWarning)



  xgb_baseline
  AUC-ROC:            0.8925
  AUC-PR:             0.4956
  Precision:          0.7917
  Recall:             0.3105
  F1:                 0.4461
  False positive rate:0.0029
  Confusion matrix:
    TP=1,262  FP=332
    FN=2,802  TN=113,712

Results logged to data/model_results/model_log.json


In [5]:
from src.models.train import tune_xgboost

tuned_xgb, study = tune_xgboost(X_train, Y_train, X_val, Y_val, n_trials=50)
print("Best validation PR-AUC:", study.best_value)
print("Best params:", study.best_params)

[I 2026-11-22 21:57:51,986] A new study created in memory with name: no-name-411eeb91-7266-4eb1-aaba-cb93259155a0
[I 2026-11-22 21:58:38,400] Trial 0 finished with value: 0.6255139811337557 and parameters: {'n_estimators': 500, 'max_depth': 10, 'learning_rate': 0.1205712628744377, 'subsample': 0.7993292420985183, 'colsample_bytree': 0.5780093202212182, 'gamma': 1.7699302940633311e-07, 'min_child_weight': 1, 'reg_alpha': 0.6245760287469893, 'reg_lambda': 0.002570603566117598}. Best is trial 0 with value: 0.6255139811337557.
[I 2026-11-22 21:59:08,120] Trial 1 finished with value: 0.597394957654187 and parameters: {'n_estimators': 767, 'max_depth': 3, 'learning_rate': 0.2708160864249968, 'subsample': 0.9162213204002109, 'colsample_bytree': 0.6061695553391381, 'gamma': 2.8483918709107956e-07, 'min_child_weight': 2, 'reg_alpha': 5.472429642032198e-06, 'reg_lambda': 0.00052821153945323}. Best is trial 0 with value: 0.6255139811337557.
[I 2026-11-22 21:59:37,899] Trial 2 finished with value:

Best validation PR-AUC: 0.6392032963695125
Best params: {'n_estimators': 669, 'max_depth': 9, 'learning_rate': 0.044940072153474946, 'subsample': 0.8688492119165147, 'colsample_bytree': 0.8868094991762521, 'gamma': 7.082230202629053e-08, 'min_child_weight': 4, 'reg_alpha': 2.1405074122175925e-06, 'reg_lambda': 1.023544851627758}


In [6]:
# Threshold comes from validation only; test is scored once with it.
best_threshold = select_threshold(Y_val, tuned_xgb.predict_proba(X_val)[:, 1])
print(f"Threshold chosen on validation: {best_threshold:.4f}")

tuned_results = evaluate_model(tuned_xgb, X_test, Y_test, model_name="xgb_tuned")
log_results(tuned_results)
tuned_results_optimal = evaluate_model(
    tuned_xgb,
    X_test,
    Y_test,
    threshold=best_threshold,
    model_name="xgb_tuned_val_threshold",
)
log_results(tuned_results_optimal)

Threshold chosen on validation: 0.1885

  xgb_tuned
  AUC-ROC:            0.8974
  AUC-PR:             0.5174
  Precision:          0.8100
  Recall:             0.3295
  F1:                 0.4684
  False positive rate:0.0028
  Confusion matrix:
    TP=1,339  FP=314
    FN=2,725  TN=113,730

Results logged to data/model_results/model_log.json

  xgb_tuned_val_threshold
  AUC-ROC:            0.8974
  AUC-PR:             0.5174
  Precision:          0.5774
  Recall:             0.4508
  F1:                 0.5063
  False positive rate:0.0118
  Confusion matrix:
    TP=1,832  FP=1,341
    FN=2,232  TN=112,703

Results logged to data/model_results/model_log.json


In [7]:
import joblib

from src.explainability.shap_explainer import build_explainer, save_explainer

os.makedirs("data/models", exist_ok=True)
joblib.dump(
    {"model": tuned_xgb, "threshold": best_threshold, "metrics": tuned_results_optimal},
    "data/models/xgboost_production.pkl",
)
save_explainer(build_explainer(tuned_xgb, X_train.sample(1000, random_state=42)))

Building TreeExplainer...
explainer built.
Explainer saved to data/models/shap_explainer.pkl


In [8]:
# Download the new artifacts, then copy them over notebooks/data/ in your local repo.
!zip -qr /content/retrained.zip data/models data/model_results
from google.colab import files

files.download("/content/retrained.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>